# 8.3 Location for a Central Warehouse

A supplier delivers to five assembly plants in the region. In the future, a
single central warehouse will supply all of them. The plants need different
numbers of truckloads per week. Management has proposed a vacant site
 in the northeast of the region. *Where should the central warehouse be
located so that the trucks drive the shortest total distance?*

In Chapter 8.1, the cost function depended on a single quantity, the order
quantity. A location, however, has two coordinates. So we extend gradient
descent to two quantities. In doing so, the derivative becomes the
**gradient**, which gives the method its name.

## How far do the trucks drive?

We place a coordinate system over the region, with $x$ pointing east and $y$
pointing north, both in kilometers. The table shows where the plants are
located and how many truckloads each plant needs per week.

| Plant | $x$ in km | $y$ in km | Loads per week |
| --- | --- | --- | --- |
| Plant 1 | 6 | 24 | 12 |
| Plant 2 | 16 | 30 | 10 |
| Plant 3 | 20 | 12 | 10 |
| Plant 4 | 8 | 6 | 8 |
| Plant 5 | 44 | 10 | 6 |

The proposed site is located at $(40, 30)$. We store the locations of the
plants in an array with one row per plant. Column 0 contains $x$, and column 1
contains $y$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# locations of the plants in km, row = [x, y]
plants = np.array([
    [ 6.0, 24.0],   # plant 1
    [16.0, 30.0],   # plant 2
    [20.0, 12.0],   # plant 3
    [ 8.0,  6.0],   # plant 4
    [44.0, 10.0],   # plant 5
])
# truckloads per week for each plant
loads = np.array([12, 10, 10, 8, 6])

# management's proposal for the central warehouse in km
proposal = np.array([40.0, 30.0])

As the cost, we count the kilometers that all trucks together drive per week.
For simplicity, we use the straight-line distance. If the warehouse is at
$(x, y)$, plant $i$ is $d_i = \sqrt{(x - x_i)^2 + (y - y_i)^2}$ away by the
Pythagorean theorem. With $n_i$ loads per week, this gives the **transport
effort**

$$E(x, y) = \sum_{i=1}^{5} n_i \cdot d_i$$

in truck-kilometers per week. We only count the trips to the plants. The
return trips double the effort but do not change the best location.

The function `effort()` receives the location as an array `[x, y]`. If we
subtract it from `plants`, NumPy subtracts it from each row. This gives us the
coordinate differences for all five plants at once.

In [ ]:
def effort(location):
    """Transport effort in truck-km per week for a warehouse at [x, y]."""
    difference = plants - location
    # TODO: ???   compute the straight-line distance to each plant with the Pythagorean theorem and store it in distances
    return np.sum(loads * distances)

effort_proposal = effort(proposal)
print(f'Effort at the proposal: {effort_proposal:.1f} truck-km per week')

With management's proposal, the trucks drive $1365.8\,\text{km}$ per week.
*Can we do better?* In Chapter 8.1, we plotted the cost against the order
quantity. Here, the effort depends on two quantities, so it forms a landscape
of hills and valleys over the map. As on a topographic map, we draw this
landscape with **contour lines**, i.e. lines along which the effort is
constant.

To do this, we evaluate `effort()` on a grid with one point every
$0.5\,\text{km}$. We compute the effort for each value of $y$ and each value of
$x$ in two nested loops and store the result in a matrix. Note that the rows
of the matrix correspond to $y$ and the columns to $x$, just like on a map.

In [ ]:
# grid over the region: x from 0 to 50 km, y from 0 to 35 km
x_values = np.linspace(0, 50, 101)
y_values = np.linspace(0, 35, 71)

# effort at each grid point, row = y, column = x
effort_values = np.zeros((len(y_values), len(x_values)))
for i in range(len(y_values)):
    for j in range(len(x_values)):
        location = np.array([x_values[j], y_values[i]])
        effort_values[i, j] = effort(location)

`ax.contour()` draws the contour lines. It takes the values of $x$, the values
of $y$ and the matrix. With `levels`, we specify the values at which lines are
drawn, and `ax.clabel()` labels the lines with their values.

In [ ]:
# contour plot of the transport effort with plants and proposal
fig, ax = plt.subplots(figsize=(8, 6))
lines = ax.contour(x_values, y_values, effort_values,
                   levels=np.arange(650, 1500, 50))
ax.clabel(lines, fontsize=8)
ax.scatter(plants[:, 0], plants[:, 1], color='black', zorder=3)
for k in range(len(plants)):
    ax.text(plants[k, 0] + 0.8, plants[k, 1] + 0.8, f'Plant {k + 1}')
ax.scatter(proposal[0], proposal[1], marker='x', s=80, color='red',
           label='proposal')
ax.set_aspect('equal')
ax.set_xlabel('x in km')
ax.set_ylabel('y in km')
ax.set_title('Transport effort in truck-km per week')
ax.legend()
plt.show()

The contour lines enclose a valley left of center, between plants 1 to 4.
This is where the effort is smallest, below $650\,\text{km}$ per week. Toward
management's proposal, the lines get closer and closer together, so the effort
rises more and more steeply there. The valley is not in the middle of the map.
Plants 1 to 4, with 40 loads in total, pull it to the left, while plant 5,
with only 6 loads, barely pulls it to the right.

## What is a contour plot?

A **contour plot** shows a function of two quantities in the plane. Each line
connects points with the same function value. Where the lines are close
together, the function changes rapidly. If closed lines shrink around a point
and the values decrease toward the center, there is a minimum inside them.

## Where does the gradient point?

As in Chapter 8.1, we want to walk downhill from the proposal. Now, however,
we can move in any direction in the plane. So we look at the two coordinate
directions separately.

* How does the effort change if we only change $x$ and keep $y$ fixed?
* How does the effort change if we only change $y$ and keep $x$ fixed?

These two slopes are called **partial derivatives**, written
$\partial E / \partial x$ and $\partial E / \partial y$. As in Chapter 8.1, we
compute each of them with the central difference quotient. To do this, we
shift the location by `dx` either only in the $x$-direction or only in the
$y$-direction. Together, the two partial derivatives form a vector, the
**gradient**

$$\nabla E = \begin{pmatrix} \partial E / \partial x \\ \partial E / \partial y \end{pmatrix}.$$

In [ ]:
def gradient(function, point, dx):
    """Gradient of function at point = [x, y].

    Each partial derivative with the central difference quotient.
    """
    shift_x = np.array([dx, 0.0])
    shift_y = np.array([0.0, dx])
    # TODO: ???   compute the partial derivative with respect to x with the central difference quotient, shifting by shift_x, and store it in derivative_x
    # TODO: ???   compute the partial derivative with respect to y with the central difference quotient, shifting by shift_y, and store it in derivative_y
    return np.array([derivative_x, derivative_y])

At the proposed site, we get the following gradient. `np.linalg.norm()`
computes its length from the two components using the Pythagorean theorem.

In [ ]:
# gradient at management's proposal
gradient_proposal = gradient(effort, proposal, 1e-6)
length = np.linalg.norm(gradient_proposal)
print('Gradient at the proposal:', np.round(gradient_proposal, 2))
print(f'Length of the gradient: {length:.2f}')

The first component is the slope in the $x$-direction, i.e. to the east. So if
the warehouse moves one kilometer east of the proposed site, the trucks drive
about $34.5\,\text{km}$ more per week. The second component belongs to the
$y$-direction: one kilometer north, they drive about $19.5\,\text{km}$ more.
These are approximations, because the gradient describes the change right at
the proposal. Both components are positive, so we should move west and south.
The gradient points in the direction in which the effort increases fastest.
Its length of $39.6$ is the slope in exactly this direction. The **negative
gradient** therefore points in the direction of steepest descent. We draw it
as an arrow in the contour plot, using `ax.annotate()` as for the force arrows
in Chapter 4. To make the arrow fit on the map, we shorten it to a quarter of
its length.

In [ ]:
# contour plot with the negative gradient at the proposal
# TODO: ???   starting at the proposal, go a quarter of the gradient in the direction of the negative gradient and store this point in arrow_tip

fig, ax = plt.subplots(figsize=(8, 6))
lines = ax.contour(x_values, y_values, effort_values,
                   levels=np.arange(650, 1500, 50))
ax.clabel(lines, fontsize=8)
ax.scatter(plants[:, 0], plants[:, 1], color='black', zorder=3)
ax.scatter(proposal[0], proposal[1], marker='x', s=80, color='red')
ax.annotate('', xy=arrow_tip, xytext=proposal,
            arrowprops=dict(arrowstyle='-|>', color='red', lw=2))
ax.set_aspect('equal')
ax.set_xlabel('x in km')
ax.set_ylabel('y in km')
ax.set_title('Direction of steepest descent at the proposal')
plt.show()

The arrow is perpendicular to the contour lines. *Why?* Along a contour line,
the effort does not change. So it changes fastest at right angles to the line.
A hiker looking for the most direct way down into the valley also walks across
the contour lines.

## Where is the best location for the central warehouse?

We can use gradient descent from Chapter 8.1 almost unchanged. The point is
now an array `[x, y]`. Instead of `central()`, we call `gradient()`, and NumPy
evaluates the update rule

$$\begin{pmatrix} x \\ y \end{pmatrix}_\text{new}
= \begin{pmatrix} x \\ y \end{pmatrix} - \alpha \cdot \nabla E$$

for both coordinates at once. As the stopping criterion, we check the length of
the gradient. At the minimum, the gradient is zero, and its length is small
only if both components are small.

In [ ]:
def gradient_descent_2d(function, start, learning_rate, tolerance,
                        max_iterations):
    """Search for a minimum of function, starting at start = [x, y].

    Returns all visited points as an array, row = [x, y].
    """
    point = start
    history = [point]
    for k in range(max_iterations):
        slope = gradient(function, point, 1e-6)
        # stopping criterion: length of the gradient almost zero
        # TODO: ???   end the loop if the length of the gradient is smaller than the tolerance
        point = point - learning_rate * slope
        history.append(point)
    return np.array(history)

We start at the proposed site, with a learning rate of $0.2$ and a
tolerance of $0.01$.

In [ ]:
# gradient descent starting from the proposal
history = gradient_descent_2d(effort, proposal, 0.2, 0.01, 200)

# result: iterations, location and effort
n_iterations = len(history) - 1
location_best = history[-1]
effort_best = effort(location_best)
print(f'{n_iterations} iterations')
print('Best location in km:', np.round(location_best, 2))
print(f'Effort: {effort_best:.1f} truck-km per week')

After 24 iterations, the method places the warehouse at $(14.37, 18.59)$.
There, the trucks drive $619.5\,\text{km}$ per week instead of
$1365.8\,\text{km}$ with management's proposal, which is about
$750\,\text{km}$ less. We draw the path of the method in the contour plot. The
first column of `history` contains all $x$-values, and the second column all
$y$-values.

In [ ]:
# path of gradient descent in the contour plot
fig, ax = plt.subplots(figsize=(8, 6))
lines = ax.contour(x_values, y_values, effort_values,
                   levels=np.arange(650, 1500, 50))
ax.clabel(lines, fontsize=8)
ax.scatter(plants[:, 0], plants[:, 1], color='black', zorder=3)
ax.plot(history[:, 0], history[:, 1], marker='o', markersize=4, color='red',
        label='path of the method')
ax.set_aspect('equal')
ax.set_xlabel('x in km')
ax.set_ylabel('y in km')
ax.set_title('Gradient descent for the warehouse location')
ax.legend()
plt.show()

The path starts out across the contour lines and then turns into the valley.
The first steps are long because the terrain at the proposal is steep. At the
bottom of the valley, the steps get shorter and shorter, just as in
Chapter 8.1.

For comparison, the contour plot evaluated `effort()` at
$101 \cdot 71 = 7171$ grid points and still pins down the best location only
to within $0.5\,\text{km}$. Gradient descent needed only 25 gradients, i.e.
100 calls to `effort()`.

## From logistics: the center-of-gravity method

A common rule of thumb places the warehouse at the **weighted center of
gravity** of the plants. Its $x$-coordinate is $\sum n_i x_i / \sum n_i$, and
the $y$-coordinate is computed the same way. However, the center of gravity
does not minimize the sum of the distances, but the sum of the squared
distances. For the minimum of the sum of the distances, there is in general no
closed-form formula. This problem is known as the **Weber problem**, after the
economist Alfred Weber, and it is solved numerically, for example with
gradient descent.

At each plant, the effort has a sharp corner where the gradient does not
exist. If one plant needs at least half of all loads, the minimum lies exactly
at this plant, and gradient descent finds it only approximately. In our
example, the minimum lies between the plants, where the gradient is zero.

## Summary and outlook

If a cost function depends on two quantities, we display it as a contour plot.
The partial derivatives tell us how much it changes if we change only one of
the two quantities. As in Chapter 8.1, we compute them with the central
difference quotient. Together, they form the gradient. It points in the
direction of steepest ascent and is perpendicular to the contour lines.
Gradient descent moves in the direction of the negative gradient and stops as
soon as the length of the gradient is almost zero. In 24 iterations, it found
a warehouse location at which the trucks drive about $750\,\text{km}$ less
per week than with management's proposal.

In Chapter 8.4, you will use `gradient_descent_2d()` to fit a straight line
with two parameters to the measured data of a sensor. Unlike here, the two
quantities will have different units. We will see that this makes choosing
the learning rate harder.

## Mini-exercises

### Mini-exercise 1

1. Answer without code: plant 5 is far away from the others. So why is the
   valley not in the middle between all five plants?
2. The supplier could also build the warehouse right next to one of the
   plants. In a loop, compute the effort for a warehouse at each of the five
   plants. Which plant is the cheapest?
3. Answer without code, using only the contour plot: two more sites are for
   sale, at $(20, 25)$ and at $(10, 14)$. Which one is cheaper, and between
   which values does the effort lie for each of them?

In [ ]:
# code cell

### Mini-exercise 2

1. Answer without code: the gradient at a point is $(-11.8, -15.9)$. In which
   compass direction should the warehouse move?
2. Compute the gradient at $(10, 10)$ and at $(14.4, 18.6)$. What do you notice
   about the second point?
3. Answer without code: how many calls to `effort()` does one call to
   `gradient()` need? How many would it be for a location in space with three
   coordinates?

In [ ]:
# code cell

### Mini-exercise 3

1. Compute the weighted center of gravity of the plants and the effort for a
   warehouse there. How many more kilometers per week do the trucks drive than
   with the result of gradient descent?
2. Start the method at $(5, 5)$. Does it end up at the same location?
3. Answer without code: plant 5 will need twice as many loads in the future.
   In which direction does the best location move?

In [ ]:
# code cell